Appendix D - Metaheuristic optimization

*This notebook contains all the sample code in Appendix D.*

## Outline

- [Basic principles](#Basic)
- [Single-solution methods](#SS)
    - [Random search](#Random_search)
    - [Tabu search](#Tabu_search)
    - [Simulated annealing](#Simulated_annealing)
- [Population-based methods](#Population)
    - [Genetic algorithms](#Genetic)
    - [Particle swarm optimization](#Swarm)
- [A simple comparison](#Comparison)

**Metaheuristics** are general-purpose search strategies for optimization problems in which gradients are unavailable, unreliable, or too expensive to compute. They are particularly useful for nonconvex, discontinuous, discrete, mixed-variable, or black-box objectives. Unlike exact optimization methods, however, they generally provide no guarantee of finding a global optimum within a finite computational budget.

## Basic principles <a id="Basic"></a>

Consider the bounded minimization problem:
$$
\min_{\mathbf{x}\in\Omega} f(\mathbf{x}), \qquad \text{with} \qquad  \Omega=\left\{\mathbf{x}:\boldsymbol{\ell}\leq\mathbf{x}\leq\mathbf{u}\right\},
$$
where the inequalities are componentwise and the objective can be evaluated for any feasible candidate $\mathbf{x}$. In black-box optimization, the analytical expression or derivatives of $f$ need not be available.

A metaheuristic repeatedly generates and evaluates candidate solutions. Its behavior is governed by the balance between:
- **exploration**, which investigates new regions of the search space; and
- **exploitation**, which refines promising solutions already found.

Excessive exploitation can cause premature convergence to a local minimum, whereas excessive exploration wastes evaluations without sufficiently refining good candidates.

Metaheuristics can be divided into *single-solution* methods, which update one current candidate, and *population-based* methods, which evolve several candidates simultaneously. Because the algorithms are stochastic, performance should be assessed over multiple independent runs and reported together with the evaluation budget. A fixed random seed is useful for reproducibility, but conclusions should not rely on a single seed.

Constraints can be handled by generating only feasible candidates, repairing infeasible solutions, or adding a penalty to the objective. For the box constraints in previous equation, the code below uses clipping. More complex constraints require problem-specific treatment.

## Single-solution methods <a id="SS"></a>

### Random search <a id="Random_search"></a>

**Random search** independently samples candidates from the feasible region and retains the best one. It is easy to implement, naturally parallel, and provides a useful baseline. Its main limitation is poor scaling with the dimension because samples are not concentrated around promising regions.

A related method, sometimes called stochastic *hill climbing*, samples candidates in a neighborhood of the current solution and accepts only improvements. This increases exploitation but can become trapped in a local minimum.

In [1]:
import numpy as np

def random_search(objective, bounds, n_iter=5000, seed=None):
    """Minimize an objective inside box constraints."""
    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]

    best_x = rng.uniform(lower, upper)
    best_value = float(objective(best_x))
    history = [best_value]

    for _ in range(n_iter):
        candidate = rng.uniform(lower, upper)
        value = float(objective(candidate))

        if value < best_value:
            best_x = candidate.copy()
            best_value = value

        history.append(best_value)

    return best_x, best_value, np.asarray(history)

### Tabu search <a id="Tabu_search"></a>

**Tabu search** is primarily designed for discrete and combinatorial problems. Starting from a current solution, it evaluates a neighborhood and moves to the best *admissible* neighbor, even when that move temporarily worsens the objective. A short-term **tabu list** forbids recently used moves or solution attributes, thereby discouraging cycles. An **aspiration criterion** can override the tabu status when a move improves the best solution found so far.

The neighborhood, tabu representation, and aspiration rule depend strongly on the problem. For example, in a travelling-salesperson problem, a move may exchange two cities and the tabu list may store recently reversed edges. For this reason, a generic continuous implementation is less meaningful than problem-specific versions.

In [2]:
def tabu_search(
    objective,
    bounds,
    n_iter=1000,
    n_neighbors=None,
    tabu_tenure=7,
    step_scale=0.10,
    step_decay=0.5,
    stagnation_limit=50,
    min_step_scale=1e-4,
    diversification_interval=250,
    initial_x=None,
    seed=None,
):
    """Continuous Tabu Search for box-constrained minimization."""

    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]

    if np.any(upper <= lower):
        raise ValueError("Each upper bound must exceed its lower bound.")

    dimension = len(bounds)
    width = upper - lower

    if n_neighbors is None:
        n_neighbors = 2 * dimension

    if initial_x is None:
        current_x = rng.uniform(lower, upper)
    else:
        current_x = np.asarray(initial_x, dtype=float).copy()

    current_value = float(objective(current_x))

    best_x = current_x.copy()
    best_value = current_value
    history = [best_value]

    # The dictionary maps a move attribute to the
    # iteration at which its tabu status expires.
    tabu_until = {}

    step = step_scale * width
    minimum_step = min_step_scale * width

    local_stagnation = 0
    global_stagnation = 0

    for iteration in range(1, n_iter + 1):

        moves = []
        attributes = []

        # Coordinate moves.
        for coordinate in range(dimension):
            for direction in (-1, 1):
                move = np.zeros(dimension)
                move[coordinate] = (direction * step[coordinate])

                moves.append(move)
                attributes.append((coordinate, direction))

        # Additional random directions.
        extra_neighbors = max(0, n_neighbors - len(moves))

        if extra_neighbors > 0:
            random_moves = rng.normal(size=(extra_neighbors, dimension))

            norms = np.linalg.norm(random_moves, axis=1, keepdims=True)
            norms[norms == 0.0] = 1.0

            random_moves = random_moves / norms
            random_moves = random_moves * step

            for move in random_moves:
                normalized_move = move / step

                coordinate = int(np.argmax(np.abs(normalized_move)))

                direction = (1 if move[coordinate] >= 0 else -1)

                moves.append(move)
                attributes.append((coordinate, direction))

        # Optionally use only a subset of coordinate moves.
        if n_neighbors < len(moves):
            selected = rng.choice(
                len(moves),
                size=n_neighbors,
                replace=False,
            )

            moves = [moves[i] for i in selected]
            attributes = [attributes[i] for i in selected]

        candidates = []

        for move, attribute in zip(moves, attributes):
            candidate_x = np.clip(current_x + move, lower, upper)

            if np.array_equal(candidate_x, current_x):
                continue

            candidate_value = float(objective(candidate_x))

            if np.isfinite(candidate_value):
                candidates.append((candidate_value, candidate_x, attribute))

        if not candidates:
            break

        candidates.sort(key=lambda candidate: candidate[0])

        selected_candidate = None

        for (candidate_value, candidate_x, attribute) in candidates:

            is_tabu = (tabu_until.get(attribute, 0) >= iteration)

            # Aspiration criterion.
            aspiration = (candidate_value < best_value)

            if not is_tabu or aspiration:
                selected_candidate = (candidate_value, candidate_x, attribute)
                break

        # Prevent premature termination if all
        # candidates happen to be tabu.
        if selected_candidate is None:
            selected_candidate = candidates[0]

        (current_value, current_x, selected_attribute) = selected_candidate

        coordinate, direction = selected_attribute

        # Forbid the reverse of the accepted move.
        reverse_move = (coordinate, -direction)

        tabu_until[reverse_move] = (iteration + tabu_tenure)

        # Remove expired tabu entries.
        tabu_until = {
            move: expiry
            for move, expiry in tabu_until.items()
            if expiry >= iteration
        }

        if current_value < best_value:
            best_x = current_x.copy()
            best_value = current_value

            local_stagnation = 0
            global_stagnation = 0
        else:
            local_stagnation += 1
            global_stagnation += 1

        history.append(best_value)

        # Intensification with a smaller neighborhood.
        if local_stagnation >= stagnation_limit:
            step = np.maximum(step_decay * step, minimum_step)

            local_stagnation = 0
            tabu_until.clear()

        # Diversification through a random restart.
        if (
            diversification_interval is not None
            and global_stagnation
            >= diversification_interval
        ):
            current_x = rng.uniform(lower, upper)
            current_value = float(objective(current_x))

            step = step_scale * width
            local_stagnation = 0
            global_stagnation = 0
            tabu_until.clear()

            if current_value < best_value:
                best_x = current_x.copy()
                best_value = current_value
                history[-1] = best_value

    return best_x, best_value, np.asarray(history)

### Simulated annealing <a id="Simulated_annealing"></a>

**Simulated annealing** augments local search by occasionally accepting a worse candidate. Let $\Delta=f(\mathbf{y}) - f(\mathbf{x})$ be the change produced by moving from the current solution $\mathbf{x}$ to a neighbor $\mathbf{y}$. The candidate is accepted with probability:
$$
P(\text{accept}) =
\begin{cases}
1, & \Delta\leq 0,\\[2mm]
\exp\left(-\dfrac{\Delta}{T}\right), & \Delta>0,
\end{cases}
$$
where $T>0$ parameter is called the *temperature*. According to the **Metropolis' criterion**, the newly produced solution is accepted as the current one if $P(\text{accept}) \geq \delta$, where $\delta$ is a random number generated from a uniform distribution in $[0, 1]$. 

At high temperature, worsening moves are relatively frequent and encourage exploration. As the temperature decreases, the algorithm becomes increasingly greedy. A common practical schedule is:
$$
T_{k+1}=\alpha T_k, \qquad 0<\alpha<1.
$$
Although global-convergence results exist for restrictive and extremely slow cooling schedules, this geometric schedule should be regarded as a practical heuristic.

In [3]:
def simulated_annealing(
    objective,
    bounds,
    n_iter=5000,
    initial_temp=1.0,
    cooling=0.995,
    step_scale=0.10,
    seed=None,
):
    """Minimize an objective by simulated annealing."""

    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]
    width = upper - lower

    current_x = rng.uniform(lower, upper)
    current_value = float(objective(current_x))

    best_x = current_x.copy()
    best_value = current_value
    history = [best_value]
    temperature = float(initial_temp)

    for _ in range(n_iter):
        step = rng.normal(0.0, step_scale * width)
        candidate = np.clip(current_x + step, lower, upper)
        candidate_value = float(objective(candidate))

        delta = candidate_value - current_value
        accept = delta <= 0

        if not accept:
            accept = rng.random() < np.exp(-delta / temperature)

        if accept:
            current_x = candidate
            current_value = candidate_value

            if current_value < best_value:
                best_x = current_x.copy()
                best_value = current_value

        history.append(best_value)
        temperature = max(cooling * temperature, 1e-12)

    return best_x, best_value, np.asarray(history)

## Population-based methods <a id="Population"></a>

### Genetic algorithms <a id="Genetic"></a>

A **genetic algorithm** evolves a *population* of candidate solutions. Each iteration, called a *generation*, generally includes:
1. **evaluation** of the objective for every individual\index{Individual};
2. **selection** of comparatively good parents;
3. **crossover** to combine information from two parents;
4. **mutation** to introduce new variation; and
5. **elitism** to preserve one or more of the best individuals.

Binary **chromosomes** are useful for discrete problems, whereas real-valued vectors are more natural for continuous optimization. In a real-coded genetic algorithm, arithmetic crossover can generate a child as:
$$
\mathbf{c} = \boldsymbol{\alpha}\odot\mathbf{p}_1 + \left(\mathbf{1}-\boldsymbol{\alpha}\right)\odot\mathbf{p}_2,
$$
where $\mathbf{p}_1$ and $\mathbf{p}_2$ are the parents, $\boldsymbol{\alpha}$ contains random coefficients in $[0, 1]$, and $\odot$ denotes componentwise multiplication. Mutation then perturbs selected components, commonly with additive Gaussian noise.

In [4]:
def genetic_algorithm(
    objective,
    bounds,
    population_size=40,
    generations=200,
    tournament_size=3,
    mutation_rate=0.10,
    mutation_scale=0.05,
    elite_size=2,
    seed=None,
):
    """Real-coded genetic algorithm for box-constrained minimization."""

    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]
    width = upper - lower
    dimension = len(bounds)

    population = rng.uniform(lower, upper, size=(population_size, dimension))

    def evaluate(pop):
        return np.asarray([objective(x) for x in pop], dtype=float)

    def select_parent(values):
        indices = rng.integers(0, population_size, size=tournament_size)
        return population[indices[np.argmin(values[indices])]]

    values = evaluate(population)
    history = []

    for _ in range(generations):
        order = np.argsort(values)
        elites = population[order[:elite_size]].copy()
        offspring = []

        while len(offspring) < population_size - elite_size:
            parent_1 = select_parent(values)
            parent_2 = select_parent(values)

            alpha = rng.random(dimension)
            child = alpha * parent_1 + (1.0 - alpha) * parent_2

            mask = rng.random(dimension) < mutation_rate
            noise = rng.normal(0.0, mutation_scale * width)
            child[mask] += noise[mask]
            child = np.clip(child, lower, upper)
            offspring.append(child)

        population = np.vstack((elites, np.asarray(offspring)))
        values = evaluate(population)
        history.append(float(np.min(values)))

    best_index = int(np.argmin(values))
    return (
        population[best_index].copy(),
        float(values[best_index]),
        np.asarray(history),
    )

### Particle swarm optimization <a id="Swarm"></a>

**Particle swarm optimization** represents each candidate as a particle with position $\mathbf{x}_i$ and velocity $\mathbf{v}_i$. Each particle remembers its personal best position $\mathbf{p}_i$, while the swarm shares the best position $\mathbf{g}$ found by any particle. The updates are:
$$
\mathbf{v}_i^{(k+1)} = \omega\mathbf{v}_i^{(k)} + c_1\mathbf{r}_{1,i} \odot \left(\mathbf{p}_i-\mathbf{x}_i^{(k)}\right) + c_2\mathbf{r}_{2,i}\odot
\left(\mathbf{g}-\mathbf{x}_i^{(k)}\right),
$$
$$
\mathbf{x}_i^{(k+1)} = \mathbf{x}_i^{(k)}+\mathbf{v}_i^{(k+1)},
$$
where $\omega$ controls inertia, $c_1$ and $c_2$ control attraction toward the personal and global best positions, and the entries of $\mathbf{r}_{1,i}$ and $\mathbf{r}_{2,i}$ are independently sampled from $\mathcal{U}(0,1)$.

In [5]:
def particle_swarm(
    objective,
    bounds,
    n_particles=30,
    n_iter=200,
    inertia=0.7,
    cognitive=1.5,
    social=1.5,
    seed=None,
):
    """Particle swarm optimizer for box-constrained problems."""
    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]
    width = upper - lower
    dimension = len(bounds)

    positions = rng.uniform(
        lower, upper, size=(n_particles, dimension)
    )
    velocities = rng.uniform(
        -0.1 * width,
        0.1 * width,
        size=(n_particles, dimension),
    )

    values = np.asarray([objective(x) for x in positions])
    personal_best = positions.copy()
    personal_values = values.copy()

    best_index = int(np.argmin(personal_values))
    global_best = personal_best[best_index].copy()
    global_value = float(personal_values[best_index])
    history = [global_value]

    for _ in range(n_iter):
        r1 = rng.random((n_particles, dimension))
        r2 = rng.random((n_particles, dimension))

        velocities = (
            inertia * velocities
            + cognitive * r1 * (personal_best - positions)
            + social * r2 * (global_best - positions)
        )

        positions = np.clip(positions + velocities, lower, upper)
        values = np.asarray([objective(x) for x in positions])

        improved = values < personal_values
        personal_best[improved] = positions[improved]
        personal_values[improved] = values[improved]

        best_index = int(np.argmin(personal_values))
        if personal_values[best_index] < global_value:
            global_best = personal_best[best_index].copy()
            global_value = float(personal_values[best_index])

        history.append(global_value)

    return global_best, global_value, np.asarray(history)

## A simple comparison <a id="Comparison"></a>

The ***Rastrigin** function* is a nonconvex $d$-dimensional benchmark with many local minima:
$$
f(\mathbf{x}) = 10d + \sum_{j=1}^{d}\left[x_j^2-10\cos(2\pi x_j)\right].
$$
Its global minimum is $f(\mathbf{0})=0$. The following code applies the preceding implementations to the two-dimensional problem.

In [6]:
def rastrigin(x):
    x = np.asarray(x, dtype=float)
    return 10.0 * x.size + np.sum(x**2 - 10.0 * np.cos(2.0 * np.pi * x))


bounds = [(-5.12, 5.12), (-5.12, 5.12)]

methods = {
    "Random search": lambda: random_search(
        rastrigin, bounds, n_iter=10000, seed=1
    ),
    "Simulated annealing": lambda: simulated_annealing(
        rastrigin,
        bounds,
        n_iter=10000,
        initial_temp=10.0,
        cooling=0.999,
        step_scale=0.05,
        seed=1,
    ),
    "Genetic algorithm": lambda: genetic_algorithm(
        rastrigin,
        bounds,
        population_size=50,
        generations=250,
        seed=1,
    ),
    "Particle swarm": lambda: particle_swarm(
        rastrigin,
        bounds,
        n_particles=40,
        n_iter=250,
        seed=1,
    ),
}

for name, run in methods.items():
    best_x, best_value, history = run()
    print(f"{name:20s}: f(x) = {best_value:.6f}, x = {best_x}")

Random search       : f(x) = 0.643764, x = [0.03096412 0.04802598]
Simulated annealing : f(x) = 0.007461, x = [ 0.0002016  -0.00612972]
Genetic algorithm   : f(x) = 0.000001, x = [1.28657533e-09 7.00865510e-05]
Particle swarm      : f(x) = 0.000000, x = [-3.52830375e-09 -4.89882807e-11]


A single run is not sufficient for ranking stochastic optimizers. A fair comparison should use the same objective-evaluation budget, several independent seeds, and summary statistics such as the median, interquartile range, and success rate. Hyperparameters should be selected without using the final test instances.